<a href="https://colab.research.google.com/github/Maddox159-crypto/2026_ESAA_assignment/blob/main/%ED%95%84%EC%82%AC34.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[개념]

### 1. 딥러닝 용어
- **가중치**: 노드 간 연결 강도. 입력 값이 결과에 미치는 영향력을 조절
- **바이어스**: 가중합에 더하는 상수로, 최종 출력 값을 조절
- **가중합(전달 함수)**: 입력과 가중치의 곱을 모두 더한 값
- **활성화 함수**: 가중합을 받아 비선형적으로 변환해 출력
  - 시그모이드: 출력 0~1. 층이 깊어지면 기울기 소멸 문제가 생김
  - 하이퍼볼릭 탄젠트: 출력 -1~1. 시그모이드의 양수 편향은 해결했지만 기울기 소멸은 여전
  - 렐루: 음수 → 0, 양수 → x. 학습이 빠르고 기울기 소멸이 없어 은닉층에 주로 사용
  - 리키 렐루: 음수일 때 아주 작은 값을 출력해 렐루의 단점 보완
  - 소프트맥스: 출력을 0~1로 만들고 합이 1이 되게 함 → 출력층(분류)에 사용
- **손실 함수**: 예측 값과 실제 값의 차이를 측정 (0에 가까울수록 좋음)
  - MSE(평균 제곱 오차): 주로 회귀에 사용
  - 크로스 엔트로피 오차: 주로 분류에 사용. MSE보다 학습이 빠름

### 2. 딥러닝 학습
- **순전파**: 입력층 → 은닉층 → 출력층 순서로 가중합과 활성화 함수를 거쳐 예측 값 계산
- **손실 계산**: 손실 함수로 예측 값과 실제 값의 오차를 구함
- **역전파**: 오차를 출력층 → 은닉층 → 입력층으로 거꾸로 전파하며, 오차를 각 가중치로 미분해 가중치를 업데이트
- 업데이트된 가중치로 다시 순전파 → 이 과정을 반복해 손실을 줄여 나감

### 3. 딥러닝의 문제점과 해결
- **과적합**: 훈련 데이터를 과하게 학습해 새로운 데이터에서 오차가 커지는 현상
  - 해결: 드롭아웃 (학습 중 일부 노드를 임의로 제외)
- **기울기 소멸**: 역전파 과정에서 오차가 점점 작아져 앞쪽 층이 학습되지 않는 현상
  - 해결: 시그모이드·tanh 대신 렐루 사용
- **성능 저하**: 경사 하강법이 느리거나 불안정한 문제
  - 배치 경사 하강법: 전체 데이터로 업데이트 → 안정적이지만 느림
  - 확률적 경사 하강법(SGD): 데이터 하나씩 업데이트 → 빠르지만 불안정
  - 미니 배치 경사 하강법: 작은 묶음 단위로 업데이트 → 빠르고 안정적이라 가장 많이 사용
- **옵티마이저**: 학습 속도와 방향을 조절해 SGD의 불안정함을 보완
  - 속도 조정: Adagrad, Adadelta, RMSProp
  - 운동량 조정: Momentum, NAG
  - 둘 다 결합: Adam (현재 가장 보편적)

[필사]

In [1]:
import torch

In [2]:
class Net(torch.nn.Module):
  def __init__(self, n_feature, n_hidden, n_output):
    super(Net, self).__init__()
    self.hidden = torch.nn.Linear(n_feature, n_hidden)
    self.relu = torch.nn.ReLu(inplace=True)
    self.out = torch.nn.Linear(n_hidden, n_output)
    self.softmax = torch.nn.Softmax(dim=n_output)
  def forward(self, x):
      x = self.hidden(x)
      x = self.relu(x)
      x = self.out(x)
      x = self.softmax(x)
      return x

In [ ]:
#import torch
#loss_fn = torch.nn.MSELoss(reduction= 'sum')
#y_pred = model(x)
#loss = loss_fn(y_pred, y)

In [ ]:
#loss = nn.CrossEntropyLoss()
#input = torch.randn(5, 6, requires_grad=True)
#target = torch.empty(3, dtype=torch.long).random_(5)
#output = loss(input, target)
#output.backward()

In [8]:
class DropoutModel(torch.nn.Module):
    def __init__(self):
        super(DropoutModel, self).__init__()
        self.layer1 = torch.nn.Linear(784, 1200)
        self.dropout1 = torch.nn.Dropout(0.5)
        self.layer2 = torch.nn.Linear(1200, 1200)
        self.dropout2 = torch.nn.Dropout(0.5)
        self.layer3 = torch.nn.Linear(1200, 10)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = self.dropout1(x)
        x = F.relu(self.layer2(x))
        x = self.dropout2(x)
        x = self.layer3(x)
        return x

In [ ]:
#class CustomDataset(Dataset):
#    def __init__(self):
#        self.x_data = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
#        self.y_data = [[12], [18], [11]]
#    def __len__(self):
#        return len(self.x_data)
#    def __getitem__(self, idx):
#        x = torch.FloatTensor(self.x_data[idx])
#        y = torch.FloatTensor(self.y_data[idx])
#        return x, y
#dataset = CustomDataset()
#dataloader = DataLoader(
#    dataset,  # 데이터셋
#    batch_size=2,  # 미니 배치 크기로 2의 제곱수를 사용하겠다는 의미
#    shuffle=True,  # 데이터를 불러올 때마다 랜덤으로 섞어서 가져옴
#)

In [ ]:
## Adagrad
#optimizer = torch.optim.Adagrad(model.parameters(), lr=0.01)  # 학습률 기본값은 1e-2

## Adadelta
#optimizer = torch.optim.Adadelta(model.parameters(), lr=1.0)  # 학습률 기본값은 1.0

## RMSProp
#optimizer = torch.optim.RMSprop(model.parameters(), lr=0.01)  # 학습률 기본값은 1e-2

## Momentum
#optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

## NAG (Nesterov Accelerated Gradient)
#optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9,
#                            nesterov=True)  # nesterov 기본값은 False

## Adam
#optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # 학습률 기본값은 1e-3